# EDA — Geocoding das capitais

Esta análise lê os Parquets da Bronze e investiga estrutura, ausências, duplicações, cobertura e coordenadas. A organização segue os princípios da aula de EDA, adaptados a uma base geográfica pequena.

**Perguntas:**

1. Temos um registro para cada uma das 27 capitais?
2. Existem campos ausentes, IDs repetidos ou coordenadas inválidas?
3. Os IDs, nomes e UFs correspondem aos registros do IBGE?
4. Que distribuição geográfica aparece nas coordenadas?

A análise não consulta APIs nem grava ou altera a Bronze. Qualquer correção aprovada será implementada na Silver.


## 1. Bibliotecas e caminhos

Use o kernel do ambiente virtual do projeto. Execute a partir da raiz do projeto ou de uma subpasta. O gráfico usa Matplotlib; os cálculos usam Spark, sem converter a base para pandas.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [ ]:
project_directory = Path.cwd().resolve()

while not (project_directory / "pipeline" / "bronze").is_dir() or not (project_directory / "data").is_dir():

    if project_directory == project_directory.parent:
        raise FileNotFoundError("Execute o notebook a partir de uma pasta dentro do projeto.")

    project_directory = project_directory.parent

BRONZE_DIRECTORY = project_directory / "data" / "bronze"
GEOCODING_DIRECTORY = BRONZE_DIRECTORY / "open_meteo" / "geocoding"
MUNICIPALITIES_DIRECTORY = BRONZE_DIRECTORY / "ibge" / "municipalities"
local_filesystem_jar = project_directory / "pipeline" / "bronze" / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"

for required_path in [GEOCODING_DIRECTORY, MUNICIPALITIES_DIRECTORY, local_filesystem_jar]:

    if not required_path.exists():
        raise FileNotFoundError(f"Caminho necessário não encontrado: {required_path}")

print(f"Geocoding: {GEOCODING_DIRECTORY}")
print(f"Referência IBGE: {MUNICIPALITIES_DIRECTORY}")


## 2. Sessão Spark

Mantemos duas tarefas simultâneas e a configuração Java que já foi validada na Bronze. Inicie este notebook com um kernel novo para que a configuração seja aplicada desde a criação do Spark.


In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Viajando com o dado - EDA Geocoding")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.driver.extraClassPath", str(local_filesystem_jar))
    .config("spark.executor.extraClassPath", str(local_filesystem_jar))
    .config("spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Versão do Spark: {spark.version}")


## 3. Leitura da Bronze

Como as pastas têm nomes simples, como `SP`, a leitura é recursiva. A UF já está armazenada na coluna `state_code` dos Parquets.


In [ ]:
coordinates = (
    spark.read
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.parquet")
    .parquet(GEOCODING_DIRECTORY.resolve().as_posix())
    .cache()
)

municipalities = (
    spark.read
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.parquet")
    .parquet(MUNICIPALITIES_DIRECTORY.resolve().as_posix())
)


## 4. Princípio estrutural

Latitude e longitude são coordenadas em graus. O ID do município é um identificador, mesmo sendo numérico; calcular sua média não tem interpretação útil. `source_file` identifica o JSON usado na coleta, preservado como metadado na Bronze.

**Verificar:** quantidade de registros, tipos, colunas e exemplos. O escopo esperado contém 27 registros, um por capital.


In [ ]:
record_count = coordinates.count()
column_count = len(coordinates.columns)

print(f"Registros: {record_count}")
print(f"Colunas: {column_count}")

coordinates.printSchema()
coordinates.orderBy("state_code").show(30, truncate=False)


## 5. Dados ausentes

Contamos `null`, textos vazios ou compostos de espaços e `NaN` nas colunas decimais. O percentual é calculado sobre o total de registros. Zero de latitude ou longitude, isoladamente, não representa ausência.

**Pergunta:** quais campos precisam de investigação antes de usar as coordenadas? Nenhum valor será preenchido automaticamente.


In [ ]:
missing_expressions = []

for field in coordinates.schema.fields:
    missing_condition = F.col(field.name).isNull()

    if field.dataType.simpleString() == "string":
        missing_condition = missing_condition | (F.trim(F.col(field.name)) == "")

    if field.dataType.simpleString() in ["float", "double"]:
        missing_condition = missing_condition | F.isnan(F.col(field.name))

    missing_expressions.append(
        F.sum(F.when(missing_condition, 1).otherwise(0)).alias(field.name)
    )

missing_totals = coordinates.agg(*missing_expressions).first()
missing_records = []
missing_cell_count = 0

for column_name in coordinates.columns:
    missing_count = int(missing_totals[column_name] or 0)
    missing_percent = 0.0

    if record_count > 0:
        missing_percent = round(100.0 * missing_count / record_count, 2)

    missing_records.append((column_name, missing_count, missing_percent))
    missing_cell_count += missing_count

missing_summary = spark.createDataFrame(
    missing_records,
    "column_name string, missing_count long, missing_percent double",
)

missing_summary.orderBy(F.desc("missing_count"), "column_name").show(truncate=False)


## 6. Duplicações e chaves

Avaliamos duplicatas exatas e repetições por município e por UF. Para o escopo atual, esperamos uma capital por UF e um registro por município.

Pontos com as mesmas coordenadas também são apresentados para investigação; essa coincidência, por si só, não autoriza remover registros.


In [ ]:
exact_duplicate_count = record_count - coordinates.dropDuplicates().count()

repeated_ids = (
    coordinates.filter(F.col("municipality_id").isNotNull())
    .groupBy("municipality_id")
    .count()
    .filter(F.col("count") > 1)
)

repeated_states = (
    coordinates.filter(F.col("state_code").isNotNull())
    .groupBy("state_code")
    .count()
    .filter(F.col("count") > 1)
)

repeated_points = (
    coordinates.filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .groupBy("latitude", "longitude")
    .agg(F.count("*").alias("record_count"), F.collect_set("state_code").alias("states"))
    .filter(F.col("record_count") > 1)
)

duplicate_id_count = repeated_ids.count()
duplicate_state_count = repeated_states.count()
repeated_point_count = repeated_points.count()

print(f"Registros extras por duplicação exata: {exact_duplicate_count}")
print(f"IDs repetidos: {duplicate_id_count}")
print(f"UFs com mais de um registro: {duplicate_state_count}")
print(f"Pontos compartilhados por vários registros: {repeated_point_count}")

repeated_ids.show(truncate=False)
repeated_states.show(truncate=False)
repeated_points.show(truncate=False)


## 7. Validade das coordenadas

Latitude deve estar entre -90 e 90; longitude, entre -180 e 180. Ausências e `NaN` também tornam o par inválido para uso geográfico.

O ponto `(0, 0)` é sinalizado separadamente por ser suspeito para uma capital brasileira. Estar dentro dos limites globais **não comprova** que o ponto esteja dentro do município ou da UF correta; essa validação exigiria limites geográficos de referência.


In [ ]:
coordinates_checked = coordinates.withColumn(
    "latitude_valid",
    F.coalesce(F.col("latitude").between(-90, 90) & ~F.isnan("latitude"), F.lit(False)),
)

coordinates_checked = coordinates_checked.withColumn(
    "longitude_valid",
    F.coalesce(F.col("longitude").between(-180, 180) & ~F.isnan("longitude"), F.lit(False)),
)

invalid_coordinates = coordinates_checked.filter(~F.col("latitude_valid") | ~F.col("longitude_valid"))
origin_points = coordinates_checked.filter((F.col("latitude") == 0) & (F.col("longitude") == 0))

invalid_coordinate_count = invalid_coordinates.count()
origin_point_count = origin_points.count()

print(f"Registros com coordenadas ausentes ou fora dos limites: {invalid_coordinate_count}")
print(f"Registros no ponto (0, 0): {origin_point_count}")

invalid_coordinates.show(truncate=False)
origin_points.show(truncate=False)


## 8. Cobertura das 27 UFs

A lista esperada inclui os 26 estados e o Distrito Federal. Comparamos as UFs presentes com essa lista, exibindo ausências, códigos inesperados e a quantidade de registros por UF.


In [ ]:
EXPECTED_STATES = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO",
    "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR",
    "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO",
]

expected_state_records = []

for state_code in EXPECTED_STATES:
    expected_state_records.append((state_code,))

expected_states = spark.createDataFrame(expected_state_records, "state_code string")
actual_states = coordinates.select("state_code").distinct()

missing_states = expected_states.join(actual_states, "state_code", "left_anti")
unexpected_states = actual_states.join(expected_states, "state_code", "left_anti")

missing_state_count = missing_states.count()
unexpected_state_count = unexpected_states.count()

coordinates.groupBy("state_code").count().orderBy("state_code").show(30, truncate=False)

print(f"UFs esperadas ausentes: {missing_state_count}")

missing_states.orderBy("state_code").show(truncate=False)

print(f"UFs inesperadas: {unexpected_state_count}")

unexpected_states.orderBy("state_code").show(truncate=False)


## 9. Coerência com o IBGE

A referência é a Bronze de municípios, cuja coleta selecionou as capitais. Comparamos ID, nome e UF; apenas na comparação de nomes ignoramos espaços nas extremidades e diferenças entre maiúsculas e minúsculas.

Essa conferência identifica rótulos divergentes e capitais ausentes. Ela não valida a localização física do ponto, pois o nome e o ID foram associados na coleta.


In [ ]:
capital_reference = municipalities.select(
    F.col("id").alias("reference_id"),
    F.col("nome").alias("reference_name"),
    F.col("state_code").alias("reference_state_code"),
).distinct()

identity_check = coordinates.join(
    capital_reference,
    coordinates["municipality_id"] == capital_reference["reference_id"],
    "left",
)

name_matches = F.upper(F.trim(F.col("name"))).eqNullSafe(F.upper(F.trim(F.col("reference_name"))))
state_matches = F.col("state_code").eqNullSafe(F.col("reference_state_code"))

identity_issues = identity_check.filter(F.col("reference_id").isNull() | ~name_matches | ~state_matches)
missing_capitals = capital_reference.join(
    coordinates.select("municipality_id").distinct(),
    capital_reference["reference_id"] == coordinates["municipality_id"],
    "left_anti",
)

identity_issue_count = identity_issues.count()
missing_capital_count = missing_capitals.count()

print(f"Capitais na referência IBGE: {capital_reference.count()}")
print(f"Registros com divergências em ID, nome ou UF: {identity_issue_count}")

identity_issues.select("municipality_id", "name", "state_code", "reference_id", "reference_name", "reference_state_code").show(truncate=False)

print(f"Capitais do IBGE sem registro de coordenadas: {missing_capital_count}")
missing_capitals.show(truncate=False)


## 10. Princípio distribucional

Os resumos descrevem a extensão e a distribuição das coordenadas válidas. Uma latitude ou longitude extrema pode simplesmente pertencer a uma capital no extremo do país; não será classificada como erro pela regra do IQR.

Média e quartis de coordenadas não representam concentração da população. Não há uma pergunta que justifique testes de normalidade nesta análise.


In [ ]:
valid_coordinates = coordinates_checked.filter(F.col("latitude_valid") & F.col("longitude_valid"))

valid_coordinates.select("latitude", "longitude").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show(truncate=False)


## 11. Dispersão geográfica

O eixo horizontal mostra longitude; o vertical, latitude. Cada ponto é identificado pela UF. O gráfico não inclui polígonos ou limites territoriais e não substitui a validação espacial.

**Pergunta:** algum ponto parece deslocado em relação às demais capitais e merece investigação? Levamos ao Python apenas os poucos registros desta fonte.


In [ ]:
plot_records = valid_coordinates.select("state_code", "latitude", "longitude").orderBy("state_code").collect()
longitude_values = []
latitude_values = []
state_labels = []

for record in plot_records:
    longitude_values.append(record["longitude"])
    latitude_values.append(record["latitude"])
    state_labels.append(record["state_code"])

if plot_records:
    figure, axis = plt.subplots(figsize=(9, 8))
    axis.scatter(longitude_values, latitude_values, color="#2d223a", s=45)

    for state_code, longitude, latitude in zip(state_labels, longitude_values, latitude_values):
        axis.annotate(state_code, (longitude, latitude), xytext=(5, 4), textcoords="offset points", fontsize=8)

    axis.set_title("Coordenadas das capitais brasileiras")
    axis.set_xlabel("Longitude (graus)")
    axis.set_ylabel("Latitude (graus)")
    axis.set_aspect("equal", adjustable="box")
    axis.grid(True, color="lightgray", alpha=0.6)
    figure.tight_layout()
    plt.show()
else:
    print("Não há coordenadas válidas para o gráfico.")


## 12. Síntese e decisões para a Silver

A síntese abaixo é calculada a partir dos resultados, sem presumir que a base esteja correta. As contagens podem se sobrepor: um registro ausente pode afetar mais de uma verificação.

**Encaminhamentos:** preservar a chave IBGE e a UF; investigar ausências, ambiguidades e divergências identificadas; validar limites geográficos quando houver uma referência espacial disponível. Nenhum registro é removido ou corrigido neste notebook.


In [ ]:
checks = [
    ("Diferença em relação aos 27 registros esperados", abs(record_count - len(EXPECTED_STATES))),
    ("Células ausentes", missing_cell_count),
    ("Duplicatas exatas adicionais", exact_duplicate_count),
    ("IDs repetidos", duplicate_id_count),
    ("UFs com múltiplos registros", duplicate_state_count),
    ("Coordenadas inválidas", invalid_coordinate_count),
    ("Pontos na origem (0, 0)", origin_point_count),
    ("Pontos repetidos para investigação", repeated_point_count),
    ("UFs esperadas ausentes", missing_state_count),
    ("UFs inesperadas", unexpected_state_count),
    ("Divergências com o IBGE", identity_issue_count),
    ("Capitais sem coordenadas", missing_capital_count),
]

has_issues = False
print(f"Registros analisados: {record_count}")

for check_name, issue_count in checks:
    print(f"{check_name}: {issue_count}")

    if issue_count > 0:
        has_issues = True

if has_issues:
    print("Há pontos para investigar antes de definir os tratamentos da Silver.")
else:
    print("Nenhum problema foi encontrado nas verificações implementadas.")

print("A localização dentro dos limites do município e da UF ainda exige validação espacial.")


## 13. Encerramento

Os Parquets permanecem inalterados. Ao encerrar a sessão, os DataFrames deixam de estar disponíveis até uma nova leitura.


In [ ]:
coordinates.unpersist()
spark.stop()
